# Seminar 3: optimization and initialization

A deep network that does not train rarely raises an error. The loss stays flat, or turns
into `nan`, or creeps down so slowly that you give up. Two decisions are behind most of
these failures, and both are made before you look at a single result:

1. **Where the weights start** (initialization).
2. **How they move** (the optimizer).

This notebook goes through both step by step, on a small MLP and FashionMNIST, so it runs
on a laptop CPU in a few minutes. It follows
[UvA DL Tutorial 4](https://uvadlc-notebooks.readthedocs.io/en/latest/tutorial_notebooks/tutorial4/Optimization_and_Initialization.html)
by Phillip Lippe. The slides for this seminar give the derivations; here we measure.

**What you will do**

| Part | Question | You will see |
|---|---|---|
| 1 | What does a good initialization keep stable? | variance of activations and gradients, layer by layer |
| 2 | Which init, for which activation? | constant, fixed σ, Xavier, Kaiming |
| 3 | What do optimizers actually do? | SGD, momentum and Adam, written from scratch |
| 4 | Where do they differ? | 2D loss surfaces, then the network from the slides, trained |
| 5 | How does this look in the repo? | `dl1/module.py` and `configs/base.yaml` |

**Running it**

```bash
uv sync --no-group cuda --group cpu            # once, as in the root README
uv run --no-sync --with jupyter --with matplotlib jupyter lab seminar_3_optim_init/
```

On Colab, PyTorch and matplotlib are already installed: just run the cells. The first run
downloads FashionMNIST (30 MB) into `data/`.

In [2]:
import copy
import math
from collections.abc import Callable
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

torch.manual_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA = Path("../data") if Path("../train.py").exists() else Path("data")  # repo root or Colab
print("device:", DEVICE)

device: cpu


## 0. Data and model

FashionMNIST: 28×28 grey images of clothes, 10 classes. We normalize the inputs to mean 0
and standard deviation 1, because every argument below starts from **inputs with unit
variance**.

In [ ]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.2861,), (0.3530,))])
full_train = datasets.FashionMNIST(DATA, train=True, download=True, transform=transform)
test_set = datasets.FashionMNIST(DATA, train=False, download=True, transform=transform)
train_set, val_set = random_split(full_train, [50_000, 10_000], torch.Generator().manual_seed(42))

train_loader = DataLoader(train_set, batch_size=1024, shuffle=True, drop_last=True)
val_loader = DataLoader(val_set, batch_size=2048)
test_loader = DataLoader(test_set, batch_size=2048)

x_probe, y_probe = next(iter(train_loader))  # one fixed batch, reused for every measurement
print(x_probe.shape, f"mean {x_probe.mean():.3f}, var {x_probe.var():.3f}")

The model is the tutorial's MLP: 784 → 512 → 256 → 256 → 128 → 10. The activation is an
argument, so the same network runs linear (`nn.Identity`), with `tanh` or with ReLU.

In [ ]:
class MLP(nn.Module):
    """A plain MLP whose activation function is a constructor argument."""

    def __init__(
        self,
        act_fn: type[nn.Module] = nn.ReLU,
        hidden: tuple[int, ...] = (512, 256, 256, 128),
        d_in: int = 784,
        n_classes: int = 10,
    ) -> None:
        """Build Linear -> act -> ... -> Linear."""
        super().__init__()
        layers: list[nn.Module] = []
        for d_out in hidden:
            layers += [nn.Linear(d_in, d_out), act_fn()]
            d_in = d_out
        layers.append(nn.Linear(d_in, n_classes))
        self.layers = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """(B, 1, 28, 28) images -> (B, n_classes) logits."""
        return self.layers(x.flatten(1))


MLP(nn.ReLU)

## 1. What we measure

Two numbers per linear layer, on one fixed batch:

- **activation variance**: the variance of each `nn.Linear` output in the forward pass;
- **gradient variance**: the variance of the gradient of each weight matrix in the backward pass.

A good initialization keeps both roughly **constant from layer to layer**. If the
activation variance shrinks, deep layers see an almost constant input. If it grows, it
explodes. If the gradient variance differs a lot between layers, no single learning rate
suits them all.

Forward hooks let us record every layer's output without touching the model.

In [ ]:
def layer_stats(model: nn.Module, x: torch.Tensor = x_probe, y: torch.Tensor = y_probe) -> dict:
    """Activation and weight-gradient variance of every nn.Linear, on one batch."""
    linears = [m for m in model.modules() if isinstance(m, nn.Linear)]
    acts: list[float] = []
    hooks = [
        m.register_forward_hook(lambda _m, _i, out: acts.append(out.var().item())) for m in linears
    ]
    model.zero_grad()
    F.cross_entropy(model(x), y).backward()
    for h in hooks:
        h.remove()
    grads = [m.weight.grad.var().item() for m in linears]
    return {"act": acts, "grad": grads}


def show(results: dict[str, dict], title: str = "") -> None:
    """Plot activation and gradient variance per layer, one line per experiment."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
    for name, r in results.items():
        layers = range(1, len(r["act"]) + 1)
        axes[0].plot(layers, r["act"], marker="o", label=name)
        axes[1].plot(layers, r["grad"], marker="o", label=name)
    for ax, what in zip(axes, ["activation variance", "weight-gradient variance"], strict=True):
        ax.set_yscale("log")
        ax.set_xlabel("linear layer")
        ax.set_title(what)
        ax.grid(alpha=0.3)
    axes[0].axhline(1.0, color="k", lw=0.8, ls="--")
    axes[1].legend()
    fig.suptitle(title)
    plt.show()


def table(results: dict[str, dict]) -> None:
    """Print the activation variances, one row per experiment."""
    for name, r in results.items():
        print(f"{name:>22}  " + "  ".join(f"{v:9.2e}" for v in r["act"]))

**In practice: how you set an init.** Write a function that touches one module and give
it to `model.apply`, which calls it on every submodule:

```python
def init_fn(m):
    if isinstance(m, nn.Linear):
        nn.init.<something>_(m.weight)
        nn.init.zeros_(m.bias)

model.apply(init_fn)
```

Every experiment below is one such function.

## 2. Initializations, from bad to good

### 2.1 A constant

All weights equal to 0.005 (not 0, which would kill every gradient). Linear network, so
only the init matters.

In [ ]:
def const_init(c: float) -> Callable[[nn.Module], None]:
    """Every weight equal to c, every bias 0."""

    def fn(m: nn.Module) -> None:
        if isinstance(m, nn.Linear):
            nn.init.constant_(m.weight, c)
            nn.init.zeros_(m.bias)

    return fn


model = MLP(nn.Identity).apply(const_init(0.005))
layer_stats(model)
for i, m in enumerate(mod for mod in model.modules() if isinstance(mod, nn.Linear)):
    distinct = m.weight.grad.unique().numel()
    print(f"layer {i + 1}: {distinct:>6} distinct gradient values out of {m.weight.numel():>7}")

Each hidden layer gets **exactly one** distinct gradient value: every weight in it
receives the same update. The neurons of a layer start equal, compute the same
thing and stay equal forever, so a layer of 256 neurons behaves like one. Symmetry has to be
broken with randomness.

### 2.2 One Gaussian, the same σ everywhere

In [ ]:
def normal_init(std: Callable[[nn.Linear], float]) -> Callable[[nn.Module], None]:
    """W ~ N(0, std(layer)^2), biases 0."""

    def fn(m: nn.Module) -> None:
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=std(m))
            nn.init.zeros_(m.bias)

    return fn


fixed = {
    f"σ = {s}": layer_stats(MLP(nn.Identity).apply(normal_init(lambda _m, s=s: s)))
    for s in (0.01, 0.1)
}
table(fixed)
show(fixed, "Linear MLP, fixed σ")

σ = 0.01 squashes the signal (the output is nearly constant); σ = 0.1 blows it up. And a σ
that happens to work here would break as soon as you change a width: the right value
depends on the **fan-in** of each layer.

### 2.3 Variance 1 / fan-in, and Xavier

For $y = Wx$ with zero-mean, independent $w$ and $x$:
$\mathrm{Var}(y_i) = d_x\,\mathrm{Var}(w)\,\mathrm{Var}(x)$. Keeping the forward variance
fixed asks for $\mathrm{Var}(w) = 1/d_x$; keeping the backward variance fixed asks for
$1/d_y$. Xavier (Glorot & Bengio, 2010) takes the harmonic mean, $2/(d_x + d_y)$.

`nn.Linear(d_in, d_out).weight` has shape `(d_out, d_in)`, so the fan-in is `shape[1]`.

In [ ]:
def fan_in(m: nn.Linear) -> float:
    """Std 1/sqrt(fan-in)."""
    return 1 / math.sqrt(m.weight.shape[1])


def xavier(m: nn.Module) -> None:
    """Xavier/Glorot normal init, as shipped in torch.nn.init."""
    if isinstance(m, nn.Linear):
        nn.init.xavier_normal_(m.weight)
        nn.init.zeros_(m.bias)


linear = {
    **fixed,
    "1 / fan-in": layer_stats(MLP(nn.Identity).apply(normal_init(fan_in))),
    "Xavier": layer_stats(MLP(nn.Identity).apply(xavier)),
}
table(linear)
show(linear, "Linear MLP")

Both keep the activations close to 1 through the whole network. Xavier also keeps the
gradient variance within one order of magnitude across layers.

### 2.4 The activation changes the answer

Xavier assumes the activation is roughly linear around 0. That holds for `tanh`. ReLU sets
the negative half to zero, which halves the second moment at every layer, so Kaiming
(He et al., 2015) doubles the variance: $\mathrm{Var}(w) = 2/d_x$.

The effect compounds with depth, so we switch to the network from the slides: 20 hidden
layers of 256 (21 linear layers). It comes back in section 4.2.

In [ ]:
def kaiming(m: nn.Module) -> None:
    """Kaiming/He normal init for ReLU, as shipped in torch.nn.init."""
    if isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
        #m.weight.data = torch.normal(torch.zeros_like(m.weight.data), torch.sqrt((2/m.weight.data.shape[1]) * torch.ones_like(m.weight.data)))
        nn.init.zeros_(m.bias)


deep = (256,) * 20
by_act = {
    "tanh + Xavier": layer_stats(MLP(nn.Tanh, deep).apply(xavier)),
    "ReLU + Xavier": layer_stats(MLP(nn.ReLU, deep).apply(xavier)),
    "ReLU + Kaiming": layer_stats(MLP(nn.ReLU, deep).apply(kaiming)),
}
show(by_act, "21 linear layers")

ReLU + Xavier loses about half the variance per layer; after 20 layers almost nothing is
left. Kaiming keeps it flat. `tanh` + Xavier settles at a lower but stable level, because
`tanh` is only linear near zero.

**What about PyTorch's default?** `nn.Linear` initializes itself with
`kaiming_uniform_(a=√5)`, which is uniform on ±1/√fan-in: a variance of 1 / (3 · fan-in),
three times smaller than our 1 / fan-in.

In [ ]:
by_act["ReLU + PyTorch default"] = layer_stats(MLP(nn.ReLU, deep))
show(by_act, "21 linear layers, with PyTorch's default")

The weight variance is a sixth of Kaiming's, so the activations shrink by about 5× per
layer until the (also random) biases are all that is left, and the gradients of the first
layers vanish to around 1e-20: those
layers do not learn at all. Fine for the 5-layer nets of the assignments; for deep plain
networks, set the init yourself. (Residual connections and normalization layers, which you will meet with
ResNets, make deep networks much less sensitive to this.)

**Cheatsheet**

| Activation | Init | `torch.nn.init` |
|---|---|---|
| linear, tanh | Xavier, $2/(d_x+d_y)$ | `xavier_normal_` / `xavier_uniform_` |
| ReLU | Kaiming, $2/d_x$ | `kaiming_normal_(nonlinearity="relu")` |
| Leaky ReLU, SELU, … | Kaiming × that activation's gain | `calculate_gain` |

## 3. Optimizers, from scratch

Every optimizer in `torch.optim` does the same two things: `zero_grad()` and `step()`. Only
`step()` differs, so we write a template and three small subclasses. Each line of `update`
is one line of the equations on the slides.

In [ ]:
class Optimizer:
    """Minimal template: subclasses implement update(p, state)."""

    def __init__(self, params: list[torch.Tensor], lr: float) -> None:
        """Keep the parameters, the learning rate and one state dict per parameter."""
        self.params = list(params)
        self.lr = lr
        self.state: list[dict] = [{} for _ in self.params]

    def zero_grad(self) -> None:
        """Forget the previous gradients."""
        for p in self.params:
            p.grad = None

    @torch.no_grad()
    def step(self) -> None:
        """Update every parameter that has a gradient."""
        for p, state in zip(self.params, self.state, strict=True):
            if p.grad is not None:
                self.update(p, state)

    def update(self, p: torch.Tensor, state: dict) -> None:
        """One parameter's update rule."""
        raise NotImplementedError


class SGD(Optimizer):
    """w <- w - lr * g."""

    def update(self, p: torch.Tensor, state: dict) -> None:
        """Step against the gradient."""
        p -= self.lr * p.grad


class SGDMomentum(Optimizer):
    """m <- b1 m + (1 - b1) g;  w <- w - lr * m."""

    def __init__(self, params: list[torch.Tensor], lr: float, momentum: float = 0.9) -> None:
        """Momentum b1 averages roughly the last 1 / (1 - b1) gradients."""
        super().__init__(params, lr)
        self.momentum = momentum

    def update(self, p: torch.Tensor, state: dict) -> None:
        """Step along a running average of the gradients."""
        g = p.grad
        state["m"] = self.momentum * state.get("m", 0) + (1 - self.momentum) * g
        p -= self.lr * state["m"]


class Adam(Optimizer):
    """Momentum plus a per-parameter step size (Kingma & Ba, 2015)."""

    def __init__(
        self,
        params: list[torch.Tensor],
        lr: float,
        beta1: float = 0.9,
        beta2: float = 0.999,
        eps: float = 1e-8,
    ) -> None:
        """The defaults are those of the paper and of torch.optim.Adam."""
        super().__init__(params, lr)
        self.beta1, self.beta2, self.eps = beta1, beta2, eps

    def update(self, p: torch.Tensor, state: dict) -> None:
        """Bias-corrected first and second moments, then a normalized step."""
        g = p.grad
        state["t"] = t = state.get("t", 0) + 1
        state["m"] = self.beta1 * state.get("m", 0) + (1 - self.beta1) * g
        state["v"] = self.beta2 * state.get("v", 0) + (1 - self.beta2) * g**2
        m_hat = state["m"] / (1 - self.beta1**t)
        v_hat = state["v"] / (1 - self.beta2**t)
        p -= self.lr * m_hat / (v_hat.sqrt() + self.eps)

**Never trust an optimizer you wrote yourself until it matches a reference.** Run ours and
PyTorch's from the same start for a few steps on the same model and data, and compare the
weights.

In [ ]:
def same_run(
    ours: type[Optimizer], theirs: type[torch.optim.Optimizer], steps: int = 5, **kw: float
) -> float:
    """Largest weight difference after `steps` updates by both optimizers."""
    a = MLP(nn.ReLU).apply(kaiming)
    b = copy.deepcopy(a)
    opt_a, opt_b = ours(a.parameters(), **kw), theirs(b.parameters(), **kw)
    for _ in range(steps):
        for model, opt in ((a, opt_a), (b, opt_b)):
            opt.zero_grad()
            F.cross_entropy(model(x_probe), y_probe).backward()
            opt.step()
    return max(
        (pa - pb).abs().max().item() for pa, pb in zip(a.parameters(), b.parameters(), strict=True)
    )


print("SGD  vs torch.optim.SGD :", same_run(SGD, torch.optim.SGD, lr=0.1))
print("Adam vs torch.optim.Adam:", same_run(Adam, torch.optim.Adam, lr=1e-3))

Differences of about 1e-6 or less are float rounding: they match. Our momentum does **not** match
`torch.optim.SGD(momentum=0.9)` exactly, and that is expected. PyTorch uses
$m \leftarrow \beta m + g$ (no $1-\beta$), which is the same rule with a learning rate
$1/(1-\beta) = 10\times$ larger. Remember that when you copy a learning rate from a paper.

## 4. Where optimizers differ

### 4.1 Two toy loss surfaces

Two functions of two weights, from the tutorial:

- **pathological curvature**: $f = \tanh^2(w_1) + 0.01\,|w_1| + \sigma(w_2)$, a narrow
  ravine, steep in $w_1$ and gentle in $w_2$;
- **steep optima**: three narrow Gaussian wells on an almost flat plain.

In [ ]:
def ravine(w: torch.Tensor) -> torch.Tensor:
    """Steep in w1, gentle in w2."""
    return torch.tanh(w[0]) ** 2 + 0.01 * w[0].abs() + torch.sigmoid(w[1])


def wells(w: torch.Tensor) -> torch.Tensor:
    """Three narrow Gaussian minima (std 0.2) on a flat plain."""
    out = torch.zeros(())
    for mx, my in ((1.0, -0.5), (-1.0, 0.5), (-0.5, -0.8)):
        out = out - torch.exp(-((w[0] - mx) ** 2 + (w[1] - my) ** 2) / (2 * 0.2**2)) / (
            2 * math.pi * 0.2**2
        )
    return out


def trajectory(
    f: Callable, opt_cls: type[Optimizer], start: tuple[float, float], steps: int = 100, **kw: float
) -> torch.Tensor:
    """Positions of one optimizer descending f from start."""
    w = torch.tensor(start, requires_grad=True)
    opt = opt_cls([w], **kw)
    path = [w.detach().clone()]
    for _ in range(steps):
        opt.zero_grad()
        f(w).backward()
        opt.step()
        path.append(w.detach().clone())
    return torch.stack(path)


def plot_paths(f: Callable, runs: dict, xr: tuple, yr: tuple, start: tuple, ax: plt.Axes) -> None:
    """Contour plot of f with one trajectory per optimizer."""
    xs, ys = torch.linspace(*xr, 300), torch.linspace(*yr, 300)
    gx, gy = torch.meshgrid(xs, ys, indexing="xy")
    ax.contourf(gx, gy, f(torch.stack([gx, gy])), levels=30, cmap="Blues_r", alpha=0.8)
    for name, (cls, kw) in runs.items():
        p = trajectory(f, cls, start, **kw)
        ax.plot(p[:, 0], p[:, 1], marker=".", ms=3, lw=1.5, label=name)
    ax.plot(*start, "k*", ms=14)
    ax.set_xlim(*xr)
    ax.set_ylim(*yr)
    ax.legend(loc="upper right")


fig, axes = plt.subplots(1, 2, figsize=(13, 6))
plot_paths(
    ravine,
    {
        "SGD, lr 10": (SGD, {"lr": 10}),
        "Momentum, lr 10": (SGDMomentum, {"lr": 10}),
        "Adam, lr 1": (Adam, {"lr": 1}),
    },
    (-5.5, 5.5),
    (-14, 5.5),
    (-5.0, 5.0),
    axes[0],
)
plot_paths(
    wells,
    {
        "SGD, lr 0.5": (SGD, {"lr": 0.5}),
        "Momentum, lr 1": (SGDMomentum, {"lr": 1}),
        "Adam, lr 0.2": (Adam, {"lr": 0.2}),
    },
    (-1.6, 1.6),
    (-1.6, 1.6),
    (0.0, 0.0),
    axes[1],
)
axes[0].set_title("Pathological curvature")
axes[1].set_title("Steep optima")
plt.show()

- **Ravine:** SGD with a large learning rate bounces from wall to wall; with a small one it
  would crawl along $w_2$. Momentum cancels the bouncing, Adam rescales each direction.
- **Wells:** gradients are tiny on the plain and huge near a minimum. SGD barely moves, then
  gets pushed off. Momentum gathers speed on the steep wall and overshoots. Adam's
  normalized step settles in a minimum.

For many more surfaces, try
[Emilien Dupont's interactive visualization](https://emiliendupont.github.io/2018/01/24/optimization-visualization/).

### 4.2 Our network, from the slides

This is the network the slides are built around: **20 hidden layers of 256 with ReLU, 21
linear layers in total**, trained on FashionMNIST with batch 256 for 2 epochs. Same seed
for every run, so only the init and the optimizer change.

To make the runs identical to the slides, we train on the full 60,000 training images,
batched from one tensor with a seeded permutation, and evaluate on the 10,000 test images.

In [ ]:
def our_network() -> MLP:
    """The network from the slides: 784 -> 20 x (Linear 256 + ReLU) -> 10."""
    return MLP(nn.ReLU, (256,) * 20)


X_train = torch.stack([x for x, _ in full_train]).flatten(1)
Y_train = full_train.targets.clone()
X_test = torch.stack([x for x, _ in test_set]).flatten(1)
Y_test = test_set.targets.clone()


def train(
    init: Callable[[nn.Module], None] | None,
    make_opt: Callable[[list], torch.optim.Optimizer],
    epochs: int = 2,
    batch_size: int = 256,
    seed: int = 0,
) -> tuple[list[float], float]:
    """Train our network; return the per-step loss and the test accuracy."""
    torch.manual_seed(seed)
    model = our_network()
    if init is not None:  # None: keep PyTorch's default init
        model.apply(init)
    model = model.to(DEVICE)
    opt = make_opt(model.parameters())
    gen = torch.Generator().manual_seed(seed)
    losses = []
    for _ in range(epochs):
        perm = torch.randperm(len(X_train), generator=gen)
        for i in range(0, len(perm) - batch_size + 1, batch_size):
            idx = perm[i : i + batch_size]
            loss = F.cross_entropy(model(X_train[idx].to(DEVICE)), Y_train[idx].to(DEVICE))
            opt.zero_grad()
            loss.backward()
            opt.step()
            losses.append(loss.item())  # a sync per step: fine on a CPU, see seminar 2
    with torch.no_grad():
        acc = (model(X_test.to(DEVICE)).argmax(1) == Y_test.to(DEVICE)).float().mean().item()
    return losses, acc

**Step 1: all defaults.** PyTorch's default init, plain SGD at lr 0.1, then the same with Adam.

In [ ]:
for name, make in {
    "SGD, lr 0.1": lambda p: torch.optim.SGD(p, lr=0.1),
    "Adam, lr 1e-3": lambda p: torch.optim.Adam(p, lr=1e-3),
}.items():
    losses, acc = train(None, make)
    print(
        f"default init + {name:<14} loss {losses[0]:.3f} -> {losses[-1]:.3f}   test acc {acc:.1%}"
    )
print(f"random guess: loss ln(10) = {math.log(10):.3f}, accuracy 10%")

Both stay at the loss of a random guess. Section 2.4 showed why: with the default init the
signal is gone after a few layers. **Step 2: Kaiming, same SGD at lr 0.1.**

In [ ]:
losses, acc = train(kaiming, lambda p: torch.optim.SGD(p, lr=0.1))
print("first 6 losses:", [round(v, 2) for v in losses[:6]])

Now the gradients arrive, and a step of 0.1 is far too big: the loss blows up to `nan`.
**Step 3: Kaiming with a smaller lr, then momentum and Adam.**

In [ ]:
runs = {
    "SGD, lr 0.01": lambda p: torch.optim.SGD(p, lr=0.01),
    "SGD + momentum 0.9, lr 0.01": lambda p: torch.optim.SGD(p, lr=0.01, momentum=0.9),
    "Adam, lr 1e-3": lambda p: torch.optim.Adam(p, lr=1e-3),
}
logs = {name: train(kaiming, make) for name, make in runs.items()}

fig, ax = plt.subplots(figsize=(10, 4))
for name, (losses, acc) in logs.items():
    ax.plot(losses, label=f"{name}  (test acc {acc:.1%})", alpha=0.85)
ax.set_xlabel("step")
ax.set_ylabel("training loss")
ax.set_ylim(0, 2.6)
ax.legend()
ax.grid(alpha=0.3)
plt.show()

Note the momentum learning rate: 0.01, the same as plain SGD's, but PyTorch's momentum makes
steps about 10× larger (section 3), which is why it starts with a bump.

Most of the improvement came from the init: from a random guess to above 80%. The optimizer
decides how fast and how smoothly. Adam usually gets going fastest with its default learning
rate; well-tuned SGD with momentum often generalizes as well or better (Wilson et al., 2017),
which is why it is still the default for CNNs like the ResNet-18 in this repo.

## 6. Try it

1. In 2.4, replace `nn.ReLU` with `nn.LeakyReLU` and fix `kaiming` with
   `nn.init.calculate_gain("leaky_relu", 0.01)`. Does the variance stay flat?
2. Make the plain-SGD run in 4.2 as fast as Adam's by tuning only the learning rate. What
   happens at lr = 1?
3. Run the toy surfaces with `torch.optim.SGD(momentum=0.9)` instead of `SGDMomentum`, at the
   same learning rate. Explain what you see with section 3.
4. Add `torch.optim.AdamW` to 4.2 with `weight_decay=0.01`. Is the test accuracy different
   after 3 epochs?

## References

- P. Lippe, [UvA DL Tutorial 4: Optimization and Initialization](https://uvadlc-notebooks.readthedocs.io/en/latest/tutorial_notebooks/tutorial4/Optimization_and_Initialization.html).
- X. Glorot & Y. Bengio (2010). Understanding the difficulty of training deep feedforward neural networks. AISTATS.
- K. He, X. Zhang, S. Ren & J. Sun (2015). Delving deep into rectifiers. ICCV.
- D. P. Kingma & J. Ba (2015). Adam: A method for stochastic optimization. ICLR.
- A. C. Wilson et al. (2017). The marginal value of adaptive gradient methods in machine learning. NeurIPS.
- P. Ouannes (2019). [How to initialize deep neural networks? Xavier and Kaiming initialization](https://pouannes.github.io/blog/initialization/).